In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
# SOURCE AND TARGET PATHS

races_source_path = f"{s3_root_path}/Silver/Aaisha/races"
circuits_source_path = f"{s3_root_path}/Silver/Aaisha/circuits"

target_path = f"{s3_root_path}/Gold/Aaisha/dim_races"

print("Races Source    :", races_source_path)
print("Circuits Source :", circuits_source_path)
print("Target Path     :", target_path)

In [0]:
circuits_df = (
    spark.read
        .format("delta")
        .load(circuits_source_path)
)

races_df = (
    spark.read
        .format("delta")
        .load(races_source_path)
)

In [0]:
dim_races_df = (
    races_df
        .join(
            circuits_df,
            races_df.circuitId == circuits_df.circuitId,
            "inner"
        )
        .select(
            races_df.season,
            races_df.round,
            races_df.raceName,
            races_df.date,
            circuits_df.circuitName,
            circuits_df.locality,
            circuits_df.country
        )
)

display(dim_races_df)

In [0]:
(
    dim_races_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

In [0]:
gold_races_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(gold_races_df)